<a href="https://colab.research.google.com/github/financieras/articulos/blob/main/FrozenLake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Aprendizaje por Refuerzo: Q-Learning con Frozen Lake**
## **Cómo aprende un agente experimentando**

## Índice

### 1. Introducción: Aprender por Ensayo y Error

* ¿Qué es el Reinforcement Learning (RL)?
* Diferencias con el Aprendizaje Supervisado (aprender de un jefe vs. aprender de la experiencia).
* El ciclo Agente ↔ Entorno ↔ Recompensa.

### 2. El Escenario: Frozen Lake y Gymnasium

* Presentación del problema: Hielo, agujeros y una meta.
* El concepto de **Entorno Estocástico**: ¿Por qué a veces el agente resbala?
* **💻 Código:** Instalación de `gymnasium` y creación del entorno.

### 3. Conceptos Clave: El Vocabulario del RL

* Estado, Acción y Recompensa.
* La Política (π): El "manual de instrucciones" que el agente debe escribir.
* El objetivo: Maximizar la recompensa acumulada.

### 4. La Q-Table: La Memoria del Agente

* ¿Qué es realmente una Q-Table? Una "hoja de trucos" de 16×4.
* **💻 Código:** Inicialización de la tabla con `numpy.zeros`.

### 5. La Ecuación de Bellman: ¿Cómo aprende el cerebro?

* Explicación intuitiva: "Actualizo lo que creía saber con lo que acabo de descubrir".
* La fórmula en LaTeX.
* Explicación de los "mandos de control": α (alfa) ritmo de aprendizaje y γ (gamma) visión a futuro.

### 6. Explorar o Explotar: El Dilema -greedy

* El dilema del agente: ¿Hago lo que ya sé que funciona o pruebo algo nuevo?
* Implementación del decaimiento de epsilon (ε-decay).

### 7. ¡A Entrenar! Enseñando al Agente a Sobrevivir

* **💻 Código:** El bucle de entrenamiento completo.
* Explicación de los pasos: `reset`, `step`, `update`, `render`.

### 8. Análisis de Resultados: ¿Ha aprendido nuestro agente?

* **💻 Código:** Gráfica de la curva de aprendizaje (recompensas por episodio).
* **💻 Código:** Visualización pro mediante un **Heatmap** de la Q-Table (¿Qué zonas considera peligrosas?).
* **💻 Código:** Demo: Ver al agente jugar.

### 9. Conclusión y Siguientes Pasos

* Limitaciones de las tablas (¿Qué pasa si el mundo es gigante?).
* Introducción al Deep Q-Learning (DQN) para triunfar en los juegos de Atari.

---


# **1. Introducción: Aprender por Ensayo y Error**

¡Bienvenido al fascinante mundo del **Reinforcement Learning (RL)** o Aprendizaje por Refuerzo! Si alguna vez has entrenado a un perro para que se siente a cambio de una golosina, o si has aprendido a montar en bicicleta cayéndote una cuantas veces antes de mantener el equilibrio, ya entiendes los principios básicos de esta rama de la Inteligencia Artificial.

### ¿Qué es el Reinforcement Learning?

A diferencia de otras áreas de la IA, el RL no se trata de predecir una etiqueta (como decir si una foto es un gato o un perro) ni de encontrar patrones ocultos en los datos. El RL trata sobre la **toma de decisiones secuenciales**.

El **Reinforcement Learning** es una rama del Machine Learning donde un **Agente** aprende a tomar decisiones interactuando con un **Entorno**. A diferencia de otros métodos de aprendizaje automático, el Agente:

- ❌ **No tiene un profesor que le diga exactamente qué hacer en cada situación**
- ✅ **Aprende de las consecuencias de sus propias acciones**
- ✅ **Recibe señales de "recompensa" o "castigo" que le indican si lo está haciendo bien o mal**

Es como aprender a jugar al ajedrez: nadie te dice "en esta posición exacta, mueve el caballo aquí". En cambio, juegas muchas partidas, pierdes, ganas, y poco a poco descubres qué estrategias funcionan mejor.

---

### Aprendizaje Supervisado vs. RL: El Jefe vs. La Experiencia

Para entenderlo mejor, comparemos el RL con el aprendizaje más tradicional:

* **Aprendizaje Supervisado (El "Jefe"):** Imagina que tienes un jefe que te da una lista de tareas y te dice exactamente cómo debe quedar cada una. Si te equivocas, te corrige al instante. Tienes ejemplos claros de "entrada" y "salida".
* **Reinforcement Learning (La "Experiencia"):** Aquí no hay jefe. Estás solo en una habitación con un objetivo. Pruebas cosas: si algo funciona, recibes una moneda; si algo sale mal, no recibes nada o recibes un pequeño "toque". Nadie te dice *qué* hacer, solo te dicen *qué tan bien* lo hiciste después de que lo intentaste.

> **En resumen:** En el aprendizaje supervisado aprendes de un "maestro". En el RL, aprendes de tu propia **experiencia**.


| **Aprendizaje Supervisado** | **Aprendizaje por Refuerzo** |
|------------------------------|------------------------------|
| **"Aprender de un profesor"** | **"Aprender de la experiencia"** |
| Tienes datos etiquetados: "esta imagen es un gato" | No hay etiquetas, solo señales de recompensa |
| El objetivo es **imitar** ejemplos correctos | El objetivo es **descubrir** la mejor estrategia |
| Aprendes de datos estáticos | Aprendes interactuando con un entorno dinámico |
| Ejemplo: Clasificar imágenes de perros y gatos | Ejemplo: Enseñar a un robot a caminar |

---


### El ciclo fundamental del RL

Un sistema de Reinforcement Learning logra que el modelo aprenda iterando este ciclo.

```
    ┌─────────────────────────────────────────┐
    │                                         │
    │   👤 AGENTE                             │
    │   (Quien toma las decisiones)           │
    │                                         │
    └──────────┬──────────────────────────────┘
               │                    ▲
               │ Acción             │ Estado +
               │                    │ Recompensa
               ▼                    │
    ┌───────────────────────────────┴─────────┐
    │                                         │
    │   🌍 ENTORNO                            │
    │   (El mundo donde actúa el agente)      │
    │                                         │
    └─────────────────────────────────────────┘
```


#### Los cinco componentes fundamentales:

1. **🤖 El AGENTE** (Agent)
   - Es quien aprende y toma decisiones
   - En nuestro caso: el algoritmo de Q-Learning
   - Piensa en él como "el jugador"

2. **🌍 El ENTORNO** (Environment)
   - Es el mundo donde actúa el agente
   - Define las reglas del juego
   - En nuestro caso: el lago congelado de Frozen Lake
   - Puede ser determinista (siempre pasa lo mismo) o estocástico (tiene aleatoriedad)

3. **📍 El ESTADO** (State - *s*)
   - Es la situación actual en la que se encuentra el agente
   - Representa "dónde estoy" o "qué está pasando ahora"
   - En Frozen Lake: La casilla específica donde está el agente
   - Ejemplo: _"Estoy en la casilla (2,3)"_

4. **🎯 La ACCIÓN** (Action - *a*)
   - Es la decisión que toma el agente
   - Representa "qué voy a hacer"
   - En Frozen Lake: Moverse en una de cuatro direcciones (←, ↓, →, ↑)
   - Ejemplo: _"Voy a moverme hacia arriba"_

5. **🎁 La RECOMPENSA** (Reward - *r*)
   - Es la señal de feedback que recibe el agente después de cada acción
   - Le dice "esto estuvo bien" (+) o "esto estuvo mal" (-)
   - En Frozen Lake: +1 por llegar a la meta, 0 en cualquier otro caso
   - Es el "profesor silencioso" que guía el aprendizaje

<img src="https://github.com/financieras/math_for_ai/blob/main/img/reinforcement_learning.jpeg?raw=1" alt="reinforcement learning" width="480"/>

---

#### ¿Cómo funciona el ciclo completo?

Ahora que conocemos los componentes, veamos cómo interactúan en cada paso:

1. **📍 El Agente observa el ESTADO actual** del entorno
   - _"Estoy en la casilla (0,0) del lago"_

2. **🎯 El Agente elige una ACCIÓN** basándose en su conocimiento actual y se ejecuta la acción.
   - _"Voy a moverme hacia la derecha"_

3. **🌍 El Entorno responde**:
   - Cambia el Estado
   - Hay un nuevo **ESTADO**: _"Ahora estás en la casilla (0,1)"_
   - Le da una **RECOMPENSA**  (puede ser positiva, negativa o cero):
   - _"0 puntos (aún no llegas a la meta)"_

4. **🤖 El Agente aprende** de esta experiencia
   - Actualiza su conocimiento: _"Ah, moverme a la derecha desde (0,0) me llevó a (0,1) sin obtener recompensa"_

5. **🔄 Se repite el ciclo** desde el paso 1 con el nuevo estado
   - El agente continúa hasta alcanzar un **estado terminal**: llegar a la meta 🎯 o caer en un agujero ❌

---

Este ciclo completo (desde el inicio hasta un estado terminal) se llama un **episodio**. El agente jugará **miles de episodios**, y con cada uno se volverá más inteligente al descubrir qué acciones llevan a mejores recompensas. Así el agente descubre una buena estrategia (lo que llamamos política).

**Piénsalo así:**
- Un episodio = Una partida completa del juego
- Miles de episodios = El entrenamiento completo del agente

---

### ¿Listo para empezar?

Un agente deberá cruzar un lago congelado evitando agujeros… ¡sin que nadie le diga por dónde ir!
- Solo recibirá una recompensa +1 cuando llegue a la meta… y -1 si cae al agua.  
- Todo lo demás lo tendrá que aprender **por ensayo y error**.


🧊 ¡Vamos a patinar sobre hielo! 🧊

---

# **2. El Escenario: Frozen Lake y Gymnasium**

Ahora que comprendes la filosofía del Reinforcement Learning, es momento de conocer el **campo de entrenamiento** donde nuestro agente aprenderá a sobrevivir: el entorno de un lago congelado lleno de peligros.

---

## 🧊 El Problema: Un Lago Peligroso

Imagina este escenario:

> *Eres un explorador que debe cruzar un lago congelado para recuperar un regalo 🎁 que está en la esquina opuesta. Empiezas en una esquina (S = Start) y necesitas llegar a donde está el regalo (G = Goal). Pero hay un problema: el hielo está agrietado y hay varios agujeros (H = Hole) que debes evitar a toda costa. Además, el hielo es resbaladizo, así que cuando intentas moverte en una dirección... ¡a veces te deslizas hacia otra!*

El mapa del lago se ve así (versión 4×4):

```
S F F F
F H F H
F F F H
H F F G
```

**Leyenda:**
- **S** (Start): Punto de inicio - Casilla segura 🚶
- **F** (Frozen): Hielo congelado - Casilla segura 🧊
- **H** (Hole): Agujero - ¡Si caes aquí, pierdes! ❌
- **G** (Goal): Meta - ¡Si llegas aquí, ganas! 🎯

**Las 4 Acciones Posibles:**
- `0` → ← Izquierda
- `1` → ↓ Abajo  
- `2` → → Derecha
- `3` → ↑ Arriba

---

## 🎲 Entorno Estocástico: El Factor Sorpresa

Aquí viene lo interesante (y frustrante): **Frozen Lake es un entorno estocástico**.

### ¿Qué significa "estocástico"?

Significa que hay **aleatoriedad** en el entorno. Cuando le dices al agente "muévete hacia la derecha →", no siempre se moverá exactamente hacia la derecha. A veces resbalará y se moverá en otra dirección.

**Piénsalo así:**
- **Entorno Determinista**: Como un tablero de ajedrez. Si mueves el caballo, sabes exactamente dónde terminará.
- **Entorno Estocástico**: Como patinar sobre hielo mojado. Intentas ir hacia la derecha, pero podrías deslizarte en cualquier dirección.

### Probabilidades de movimiento en Frozen Lake

Por defecto, cuando eliges una acción:
- **33.3%** de probabilidad: Te mueves en la dirección que elegiste
- **33.3%** de probabilidad: Te deslizas perpendicular a la izquierda
- **33.3%** de probabilidad: Te deslizas perpendicular a la derecha

Por ejemplo, si decides moverte **→ (derecha)**:
- 33.3% → Te mueves a la derecha (como querías)
- 33.3% → Te deslizas hacia arriba ↑
- 33.3% → Te deslizas hacia abajo ↓

> **¡Esto hace que el aprendizaje sea mucho más desafiante!** El agente debe aprender a lidiar con esta incertidumbre y encontrar estrategias robustas que funcionen incluso cuando las cosas no salen como esperaba.

---

## 🎯 Sistema de Recompensas

El sistema de recompensas en Frozen Lake es muy simple:

| **Evento** | **Recompensa** |
|------------|:----------------:|
| Alcanzar la meta (G) | **+1** 🎁 |
| Caer en un agujero (H) | **0** 💀 |
| Pisar hielo seguro (F) | **0** 👣 |

**Nota importante:** Aunque caer en un agujero da recompensa 0, el episodio termina inmediatamente, por lo que el agente aprenderá a evitarlos (ya que no puede seguir acumulando recompensas futuras).

---

## 🐍 ¡Manos al código! Instalación y Configuración

Vamos a usar **Gymnasium** (anteriormente conocido como OpenAI Gym), que es la biblioteca estándar para entornos de RL.

### Paso 1: Instalación

```python
# Instalamos Gymnasium
!pip install gymnasium

# Importamos las librerías necesarias
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

print("✅ Librerías importadas correctamente")
```

---

### Paso 2: Creación del Entorno

```python
# Creamos el entorno Frozen Lake
# is_slippery=True activa el modo estocástico (con resbalones)
env = gym.make(
    'FrozenLake-v1',
    map_name="4x4",      # Especifica el tamaño del mapa
    is_slippery=True,    # Activa el comportamiento estocástico
    render_mode='ansi'   # Modo de visualización en texto
)

print("🧊 Entorno Frozen Lake creado")
print(f"📊 Tamaño del espacio de estados: {env.observation_space.n}")
print(f"🎮 Tamaño del espacio de acciones: {env.action_space.n}")
```

**Salida esperada:**
```
🧊 Entorno Frozen Lake creado
📊 Tamaño del espacio de estados: 16
🎮 Tamaño del espacio de acciones: 4
```

**Explicación:**
- **16 estados**: Una casilla para cada posición del tablero 4×4
- **4 acciones**: Izquierda (0), Abajo (1), Derecha (2), Arriba (3)

---

### Paso 3: Explorando el Entorno

Vamos a ver cómo se ve el lago y entender la numeración de los estados:

```python
# Reiniciamos el entorno para empezar un nuevo episodio
state, info = env.reset()

print("🎬 Estado inicial del lago:")
print(env.render())
print(f"\n📍 Posición inicial (estado): {state}")
```

**Salida esperada:**
```
🎬 Estado inicial del lago:

SFFF
FHFH
FFFH
HFFG

📍 Posición inicial (estado): 0
```

### Numeración de Estados (Casillas del tablero)

```
 0   1   2   3      S   F   F   F
 4   5   6   7  →   F   H   F   H
 8   9  10  11      F   F   F   H
12  13  14  15      H   F   F   G
```

El estado 0 es la esquina superior izquierda (S), y el estado 15 es la esquina inferior derecha (G).

---

### Paso 4: Probando Acciones Aleatorias

Veamos qué pasa cuando el agente toma acciones al azar:

```python
# Diccionario para traducir las acciones
action_names = {0: "← Izquierda", 1: "↓ Abajo", 2: "→ Derecha", 3: "↑ Arriba"}

# Probamos 10 pasos aleatorios
env.reset()
print("🎲 Tomando 10 acciones aleatorias...\n")

for step in range(10):
    # Elegimos una acción aleatoria
    action = env.action_space.sample()
    
    # Ejecutamos la acción
    new_state, reward, terminated, truncated, info = env.step(action)
    
    # Mostramos el resultado
    print(f"Paso {step+1}:")
    print(f"  Acción elegida: {action_names[action]}")
    print(f"  Nuevo estado: {new_state}")
    print(f"  Recompensa: {reward}")
    print(f"  ¿Terminó el episodio? {terminated or truncated}\n")
    
    # Si el episodio terminó (caímos en agujero o llegamos a la meta)
    if terminated or truncated:
        print("🏁 Episodio terminado")
        if reward == 1:
            print("🎉 ¡GANAMOS! Llegamos a la meta")
        else:
            print("💀 Caímos en un agujero")
        break

# Cerramos el entorno
env.close()
```

> **💡 Observa:** El método `env.step(action)` devuelve 5 valores importantes:
> - `new_state`: El nuevo estado (casilla) donde terminó el agente
> - `reward`: La recompensa obtenida (+1 solo al llegar a la meta)
> - `terminated`: True si el episodio acabó (llegó a G o cayó en H)
> - `truncated`: True si se excedió el límite de pasos
> - `info`: Información adicional del entorno

---

## 🤔 Reflexión: ¿Por qué es difícil este problema?

Detengámonos un momento a pensar en los desafíos que enfrenta nuestro agente:

1. **🎲 Aleatoriedad**: El hielo resbaladizo hace que las acciones sean impredecibles
2. **🕳️ Agujeros mortales**: Un movimiento en falso y se acabó el episodio
3. **🎯 Recompensa diferida**: Solo obtienes +1 al llegar a la meta, no hay "pistas" en el camino
4. **🗺️ Caminos múltiples**: Hay varias rutas posibles para llegar a la meta

**Sin embargo**, nuestro agente de Q-Learning aprenderá a:
- Identificar qué acciones son más seguras en cada casilla
- Encontrar el camino óptimo considerando el riesgo de resbalones
- Maximizar la probabilidad de llegar a la meta

---

## ✅ Recapitulación

Hasta ahora hemos aprendido:

✔️ **Frozen Lake** es un lago congelado con agujeros donde debemos llegar a la meta  
✔️ Es un **entorno estocástico**: las acciones no siempre tienen el resultado esperado  
✔️ Tenemos **16 estados** (casillas) y **4 acciones** (direcciones)  
✔️ La **recompensa** solo se obtiene al llegar a la meta (+1)  
✔️ Gymnasium nos permite crear y manipular este entorno fácilmente

---

### 🚀 Siguiente Paso

Ahora que conocemos nuestro campo de batalla, es momento de profundizar en los **conceptos clave del RL**: Estado, Acción, Recompensa y, lo más importante, la **Política** que nuestro agente debe aprender.

En la siguiente sección descubriremos qué significa realmente "aprender una política" y cómo el agente decide qué hacer en cada situación.

🎯 ¡Continuemos!

---